# Import dependencies

In [1]:
import polars_bio as pb
import pandas as pd
from polars_bio.range_viz import visualize_intervals

INFO:polars_bio:Creating BioSessionContext


# Import data

In [2]:
pb.read_fastq("../../tests/resources/example.fastq")

INFO:polars_bio:Table: example registered for path: ../../tests/resources/example.fastq


In [3]:
pb.sql("show tables").collect()

0rows [00:00, ?rows/s]

table_catalog,table_schema,table_name,table_type
str,str,str,str
"""datafusion""","""public""","""example""","""BASE TABLE"""
"""datafusion""","""information_schema""","""tables""","""VIEW"""
"""datafusion""","""information_schema""","""views""","""VIEW"""
"""datafusion""","""information_schema""","""columns""","""VIEW"""
"""datafusion""","""information_schema""","""df_settings""","""VIEW"""
"""datafusion""","""information_schema""","""schemata""","""VIEW"""


In [4]:
pb.sql("select * from example limit 10").collect()

0rows [00:00, ?rows/s]

name,description,sequence,quality_scores
str,str,str,str
"""SRR9130495.1""","""D00236:723:HG32CBCX2:1:1108:13…","""NCAATACAAAAGCAATATGGGAGAAGCTAC…","""#4BDFDFFHGHGGJJJHIIIIGGIIJGJJG…"
"""SRR9130495.2""","""D00236:723:HG32CBCX2:1:1108:14…","""NGTCAAAGATAAGATCAAAAGGCACTGGCT…","""#1=DDDDD>DHFH@EFHHGHGGFGIIIGIG…"
"""SRR9130495.3""","""D00236:723:HG32CBCX2:1:1108:17…","""GTTTTCCTCTGGTTATTTCTAGGTACACTG…","""@@@DDDFFHHHFHBHIIGJIJIIJIIIEHG…"
"""SRR9130495.4""","""D00236:723:HG32CBCX2:1:1108:16…","""GGGAGGCGCCCCGACCGGCCAGGGCGTGAG…","""CCCFFFFFHHHHGHIIIGIIJIIIJJGHFF…"
"""SRR9130495.5""","""D00236:723:HG32CBCX2:1:1108:16…","""CACTCCGCCACTACAGCAGTCCCCCAGTGT…","""++=A1A:1ADA<;FFDC?;CG<F;::1CFI…"
"""SRR9130495.6""","""D00236:723:HG32CBCX2:1:1108:23…","""CGATAAAGGACTTTCAGTCAACCAACTAGA…","""CC@DDDBDFFHHHJJIJJIIJIJJJIIIHG…"
"""SRR9130495.7""","""D00236:723:HG32CBCX2:1:1108:35…","""TGGCAATGGTGTTTCTTCTTATATGATGCT…","""@CCFFFFFHHHHHIGIJJIJIJJJIJJJJJ…"
"""SRR9130495.8""","""D00236:723:HG32CBCX2:1:1108:37…","""TATTTCTTACTCTTTCAGATGTTACCCTCC…","""CCCFFFFFHHHHHJJJJGGIIJIJJHGJJJ…"
"""SRR9130495.9""","""D00236:723:HG32CBCX2:1:1108:39…","""NAAACTTTGATGTCCTAGCCCCAGGAGATG…","""#4=DFDFFHHHHHIJJJHGIIIJJJJGIIJ…"


In [5]:
pb.sql("select avg(length(sequence)) as read_length from example").collect()

0rows [00:00, ?rows/s]

read_length
f64
101.0


In [6]:
print([name for name in dir(pb) if not name.startswith("_")])

['FilterOp', 'InputFormat', 'LazyFrame', 'POLARS_BIO_MAX_THREADS', 'ReadOptions', 'VcfReadOptions', 'base_sequence_quality', 'constants', 'context', 'count_overlaps', 'coverage', 'ctx', 'describe_vcf', 'from_polars', 'interval_op_helpers', 'io', 'logging', 'merge', 'nearest', 'operations', 'overlap', 'polars_bio', 'polars_ext', 'range_op', 'range_op_helpers', 'range_op_io', 'range_viz', 'range_wrappers', 'read_bam', 'read_fasta', 'read_fastq', 'read_table', 'read_vcf', 'register_vcf', 'register_view', 'set_option', 'sql', 'visualize_intervals']


In [8]:
pb.base_sequence_quality("example")

In [9]:
pb.sql("SELECT * FROM sequence_quality_result").collect()

0rows [00:00, ?rows/s]

position,score
i64,f64
0,2.0
1,19.0
2,33.0
3,35.0
4,37.0
…,…
96,35.0
97,32.0
98,35.0


In [10]:
pb.sql("""
    SELECT
        position,
        AVG(score) AS average,
        MIN(score) AS min,
        MAX(score) AS max,
        approx_percentile_cont(score, 0.25) AS q1,
        approx_percentile_cont(score, 0.5) AS median,
        approx_percentile_cont(score, 0.75) AS q3
    FROM sequence_quality_result
    GROUP BY position
""").collect()

0rows [00:00, ?rows/s]

position,average,min,max,q1,median,q3
i64,f64,f64,f64,f64,f64,f64
0,30.135,2.0,34.0,31.0,33.083333,34.0
1,31.21,10.0,34.0,31.0,34.0,34.0
2,32.015,16.0,34.0,31.0,34.0,34.0
3,35.69,19.0,37.0,35.0,37.0,37.0
4,35.68,16.0,37.0,35.0,37.0,37.0
…,…,…,…,…,…,…
96,31.315,2.0,38.0,32.055556,34.0,35.0
97,30.67,2.0,38.0,31.0,34.0,35.0
98,31.55,7.0,40.0,31.75,34.0,35.0


In [13]:
pb.sql("""
    SELECT
        position,
       COUNT(score)
    FROM sequence_quality_result
    GROUP BY position
""").collect()

0rows [00:00, ?rows/s]

position,count(sequence_quality_result.score)
i64,i64
0,200
1,200
2,200
3,200
4,200
…,…
96,200
97,200
98,200
